# rank-2 CTRNN 训练：资源占用与并发策略微基准

日期：2026-09-04 · 机器：AMD Zen 5 8C/8T（无超线程）+ NVIDIA RTX 5070 Ti 16 GiB · torch 2.14.0+cu130

## 摘要

- 单 run 在 CUDA 上的每步延迟对 `torch` intra-op 线程数**完全不敏感**（8/4/2/1 线程 → 117/114/120/119 ms/update），
  线程数只改变 CPU 占用（515% → 97%）。默认 8 线程存在轻中度 **over-threading**。
- CPU 高占用主要发生在 **validation 段**（~23% wall：float64→float32 批量转换/拷贝、占 2–4 核）与
  **batch 生成**（~6%，numpy）；占 wall 约 70% 的 train 段实际是 ~1 核的 host/CUDA dispatch 主导。
- 单 run GPU util 仅 ~27–30%、自占显存仅 74 MiB → **不是 GPU compute bound，也不是 CPU 多核 bound，
  而是 host/CUDA dispatch + 同步瓶颈**。
- 并行 2–3 个 run 把 GPU 用到 70–77%，总吞吐 +31%～+43%（~12 updates/s 封顶，c≥3 饱和）。
- 已按此把训练配置预设为 `num_threads: 2`（见 `configs/train/rank2_baseline.yaml`），适合并发多 seed。

推荐策略：**3 个并发 seed run × 每 run 1–2 线程**；折中稳健选择 2 并发 × 2 线程/run。

## 1. 环境与测量方法

| 项 | 值 |
|---|---|
| CPU | AMD Zen 5，8 逻辑 = 8 物理核（无超线程） |
| GPU | NVIDIA GeForce RTX 5070 Ti（CC 12.0，70 SM，16 GiB） |
| torch / numpy / Python | 2.14.0+cu130 / 2.5.2 / 3.13.13 |
| 默认 `torch.get_num_threads()` / `get_num_interop_threads()` | 8 / 8 |
| 训练语义 | 复用真实 trainer：`batch_size=64`、`validation_batch_size=128`、Adam lr=1e-3、fixed seed=20260903、best.pt/metrics/periodic checkpoint 复刻 |
| 单 run 自占显存峰值 | 74 MiB（显存远未触及上限） |

方法要点：线程控制仅进程内 `torch.set_num_threads`（未动 OMP/MKL 全局默认）；单 run 每配置 500 updates + 每阶段 6 s
独立窗口采样；并发测试为真实多进程共享同一 GPU；CPU 占用 = 进程 CPU 采样均值（进程级，可 >100%），GPU util =
`nvidia-smi` 采样均值。基准脚本与原始数据位于 `%TEMP%\sm_bench\`（机器重启后可能被清空，关键数字已固化在本 notebook）。


## 2. 单 run：线程扫描（每配置 500 updates）

| CPU threads/run | ms/update | updates/s | 进程 CPU 均值 | 进程 OS 线程数 | GPU util |
|---:|---:|---:|---:|---:|---:|
| 8（默认，改前即当前配置） | 117.3 | 8.5 | 515% | ~26 | 30% |
| 4 | 113.6 | 8.8 | 359% | ~25 | 27% |
| 2 | 120.1 | 8.3 | 186% | ~21 | 27% |
| 1 | 119.1 | 8.4 | 97% | ~19 | 27% |

读取方式：**延迟对线程数几乎无感**（<5% 波动），CPU 占用却从 ~5 核降到 ~1 核 → 默认线程配置只为 ~1 核实际工作的
train 段空耗 3–4 个核。

### 各阶段归属（单 run，default 8 线程）

| 阶段 | wall 占比 | ms/iter | 进程 CPU | GPU util |
|---|---:|---:|---:|---:|
| batch 生成（numpy，每 update 一次） | ~6% | 7.1 | ~1 核 | ~7% |
| train（rollout fwd/bwd + Adam + 范数） | ~70% | 74.6 | ~1 核 | ~30% |
| validation（固定 batch rollout + 指标） | ~23% | 28.1 | 2–4 核 | ~31% |
| diagnostics（QR/SVD） | <1% | 0.75 | ~1 核 | ~22% |
| checkpoint / io | ~1% | 1.6 / 次 | ~1 核 | ~5% |

train 段每 update 约 100+ 时间步 × 每步 8–12 个微小 CUDA kernel，另有多次 `.cpu()`/同步 → 以 host dispatch 为主，
GPU 大部分时间空转（util ~30%）。

## 3. 并发测试（真实多进程共享 GPU/CPU，K=250–400 updates/run）

| concurrent runs | CPU threads/run | ms/update/run | 总吞吐 updates/s | 系统 CPU | GPU util |
|---:|---:|---:|---:|---:|---:|
| 1 | 1 | 119 | 8.4 | — | 27% |
| 2 | 1 | 178 | **11.0**（+31%） | 57% | 70% |
| 2 | 2 | 170 | **11.4**（+36%） | 74% | 70% |
| 3 | 1 | 244 | **12.0**（+43%） | 59% | 75% |
| 3 | 2 | 247 | 11.8（+40%） | 82% | 75% |
| 4 | 1 | 323 | 12.05（+43%） | 66% | 77% |

并发行是系统级 CPU%（本机空闲基线 ~23%），单 run 行是进程级，两者不可直接比。

**折算真实任务**（baseline 20000 updates/run）：顺序跑 S 个 seed ≈ 40×S 分钟；
2 并发 ≈ 57 min/2 seeds（省 ~29%）；3 并发（t1）≈ 81 min/3 seeds（省 ~32%）；4 并发吞吐不再增长。

## 4. 结论

1. **是否存在 CPU over-threading**：存在（轻中度）。默认 8 线程下进程 ~26 个 OS 线程、CPU ~5 核；降到 1–2 线程后每步延迟不变（117→119 ms），说明空耗了 3–4 个核。
2. **单 run 推荐 CPU 线程数**：**2**（与 8/4/1 延迟相同，CPU 占用从 515% 降到 186%）；只在单 run 独占机器时才考虑 4。
3. **是否值得并行多个 seed**：值得。单 run GPU util 仅 ~27–30%，并行 2–3 个可把 GPU 用到 70–76%，总吞吐 +31%～+43%，等量 seeds 总耗时缩短约 1/3。
4. **推荐最大并发 run 数**：**3**。c=3 → 4 吞吐不增长（12.0 → 12.05）而单 run 延迟从 244 线性恶化到 323 ms。
5. **主要瓶颈**：**host/CUDA dispatch + 同步**（非 GPU compute、非 CPU 多核计算）。证据：单 run GPU util ~30%、显存 74 MiB；占 wall 70% 的 train 段 ~1 核 host 绑定；并发把 GPU util 推到 ~77% 后封顶。

### 推荐运行策略（吞吐优先）

启动 3 个并发 run（不同 seed），每 run 训练线程数 ≤2；或更稳的 2 并发 × 2 线程/run。正式训练前务必以目标并发数
重跑本基准的量级验证。

### 落地（已提交仓库）

`configs/train/rank2_baseline.yaml` 顶层已预设：

```yaml
num_threads: 2   # 每进程 PyTorch intra-op 线程数（null = 保持进程默认）
```

- `TrainConfig` 新增可选 `num_threads` 字段（`src/slow_manifold/training/trainer.py`），
  `train_model` 在首个计算前调用 `torch.set_num_threads`（仅影响当前进程，interop 与全局默认不动），并写入 `run.log`。
- `metadata.yaml` 记录 `num_threads` 与 `interop_threads`，保证可追溯。
- 所有引用该 train 组件的 recipe（正式 baseline、smoke）自动生效。

## 5. 复现与原始数据

- 基准脚本：`%TEMP%\sm_bench\bench_common.py` / `run_single.py` / `run_concurrent.py` / `summarize.py`（不属仓库）。
- 原始 JSON：`%TEMP%\sm_bench\results\single\t{0,1,2,4}\result.json`
  与 `%TEMP%\sm_bench\results\concurrent2\concurrent_t*_r*.json`。
- 重跑单 run：`python run_single.py --threads T --updates 500 --phase-seconds 6 --outdir <dir>`
- 重跑并发：`python run_concurrent.py --threads T --runs R --updates K --outdir <dir>`
- 备注：测量带桌面背景负载（空闲系统 CPU ~23%、GPU 显存 ~3.2 GiB 被桌面应用占用）；<50 updates 的短跑存在预热虚高，
  表中数值均取 ≥250 updates 稳态段。本机为 8 物理核无超线程，线程/并发数字不可直接外推到其它 CPU/GPU。

In [ ]:
# 只读自检：当前进程的 torch 线程设置（运行前请先配置 kernel）
import os
import torch

print("cuda available:", torch.cuda.is_available())
print("torch intra threads:", torch.get_num_threads())
print("torch interop threads:", torch.get_num_interop_threads())
print("logical cpus:", os.cpu_count())
print("cpu count (psutil, physical):", end=" ")
try:
    import psutil

    print(psutil.cpu_count(logical=False))
except Exception:
    print("psutil not installed")